# Python Testing
Covers: unittest, pytest, fixtures, parametrize, mock/patch, side_effect, assert patterns, coverage, TDD

## 1. unittest — Built-in Framework

In [ ]:
import unittest

# Class under test
class Calculator:
    def add(self, a, b): return a + b
    def subtract(self, a, b): return a - b
    def multiply(self, a, b): return a * b
    def divide(self, a, b):
        if b == 0: raise ZeroDivisionError('Cannot divide by zero')
        return a / b
    def factorial(self, n):
        if n < 0: raise ValueError('Factorial not defined for negative numbers')
        return 1 if n == 0 else n * self.factorial(n - 1)

class TestCalculator(unittest.TestCase):
    def setUp(self):
        """Called before each test."""
        self.calc = Calculator()
    
    def test_add(self):
        self.assertEqual(self.calc.add(2, 3), 5)
        self.assertEqual(self.calc.add(-1, 1), 0)
        self.assertEqual(self.calc.add(0, 0), 0)
    
    def test_divide(self):
        self.assertAlmostEqual(self.calc.divide(10, 3), 3.333, places=3)
        self.assertEqual(self.calc.divide(10, 2), 5.0)
    
    def test_divide_by_zero(self):
        with self.assertRaises(ZeroDivisionError) as ctx:
            self.calc.divide(10, 0)
        self.assertIn('zero', str(ctx.exception).lower())
    
    def test_factorial(self):
        self.assertEqual(self.calc.factorial(0), 1)
        self.assertEqual(self.calc.factorial(5), 120)
        self.assertEqual(self.calc.factorial(10), 3628800)
    
    def test_factorial_negative(self):
        with self.assertRaises(ValueError):
            self.calc.factorial(-1)
    
    @unittest.skip('Not implemented yet')
    def test_future_feature(self):
        pass

# Run tests in notebook
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestCalculator)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

## 2. pytest — Modern Testing

In [ ]:
# pytest tests are plain functions — no class needed
# Run with: pytest test_file.py -v

# Demonstrate pytest-style assertions
import traceback

def run_test(test_func):
    """Helper to run a test function and show result."""
    try:
        test_func()
        print(f'  PASS: {test_func.__name__}')
    except AssertionError as e:
        print(f'  FAIL: {test_func.__name__}: {e}')
    except Exception as e:
        print(f'  ERROR: {test_func.__name__}: {type(e).__name__}: {e}')

calc = Calculator()

def test_add_basic():
    assert calc.add(2, 3) == 5

def test_add_negative():
    assert calc.add(-1, -2) == -3

def test_divide_normal():
    result = calc.divide(10, 2)
    assert result == 5.0, f'Expected 5.0, got {result}'

def test_divide_by_zero():
    try:
        calc.divide(10, 0)
        assert False, 'Should have raised ZeroDivisionError'
    except ZeroDivisionError:
        pass  # expected

def test_float_approx():
    # Floating point comparison
    result = 0.1 + 0.2
    assert abs(result - 0.3) < 1e-10, f'Expected ~0.3, got {result}'

print('Running pytest-style tests:')
for test in [test_add_basic, test_add_negative, test_divide_normal, 
             test_divide_by_zero, test_float_approx]:
    run_test(test)

## 3. Parametrize — Multiple Test Cases

In [ ]:
# Simulate pytest.mark.parametrize behavior

def is_palindrome(s):
    s = s.lower().replace(' ', '')
    return s == s[::-1]

def fizzbuzz(n):
    if n % 15 == 0: return 'FizzBuzz'
    if n % 3 == 0: return 'Fizz'
    if n % 5 == 0: return 'Buzz'
    return str(n)

# Parametrized test cases
palindrome_cases = [
    ('racecar', True),
    ('hello', False),
    ('A man a plan a canal Panama', True),
    ('', True),
    ('a', True),
    ('ab', False),
    ('Was it a car or a cat I saw', True),
]

print('Palindrome tests:')
all_passed = True
for text, expected in palindrome_cases:
    result = is_palindrome(text)
    status = 'PASS' if result == expected else 'FAIL'
    if status == 'FAIL': all_passed = False
    print(f'  [{status}] is_palindrome({text!r}) == {expected}')

fizzbuzz_cases = [
    (1, '1'), (2, '2'), (3, 'Fizz'), (4, '4'), (5, 'Buzz'),
    (6, 'Fizz'), (10, 'Buzz'), (15, 'FizzBuzz'), (30, 'FizzBuzz'),
]

print('\nFizzBuzz tests:')
for n, expected in fizzbuzz_cases:
    result = fizzbuzz(n)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'  [{status}] fizzbuzz({n}) = {result!r}')

# In real pytest:
# @pytest.mark.parametrize('text,expected', palindrome_cases)
# def test_palindrome(text, expected):
#     assert is_palindrome(text) == expected

## 4. Mock and Patch

In [ ]:
from unittest.mock import Mock, MagicMock, patch, call

# Basic Mock
mock = Mock()
mock.method(1, 2, 3)
mock.method('hello')

print('call_count:', mock.method.call_count)
print('call_args_list:', mock.method.call_args_list)
mock.method.assert_called_with('hello')  # last call
print('assert_called_with passed')

# Return values
mock.get_user.return_value = {'id': 1, 'name': 'Alice'}
result = mock.get_user(1)
print('\nReturn value:', result)

# MagicMock — supports magic methods
magic = MagicMock()
magic.__len__.return_value = 42
magic.__iter__.return_value = iter([1, 2, 3])
print('\nlen(magic):', len(magic))
print('list(magic):', list(magic))

# patch — replace during test
import os

with patch('os.path.exists') as mock_exists:
    mock_exists.return_value = True
    result = os.path.exists('/fake/path')
    print('\nPatched os.path.exists:', result)
    mock_exists.assert_called_once_with('/fake/path')

# Real os.path.exists restored
print('Real os.path.exists:', os.path.exists('/fake/path'))

# patch.object
class EmailService:
    def send(self, to, subject, body):
        # Would send real email
        raise RuntimeError('Real email sending disabled in tests')

service = EmailService()
with patch.object(service, 'send') as mock_send:
    mock_send.return_value = {'status': 'sent', 'id': '123'}
    result = service.send('alice@example.com', 'Hello', 'World')
    print('\nMocked send result:', result)
    mock_send.assert_called_once_with('alice@example.com', 'Hello', 'World')

## 5. side_effect and Advanced Mocking

In [ ]:
from unittest.mock import Mock, patch

# side_effect — raise exception
mock = Mock()
mock.connect.side_effect = ConnectionError('Connection refused')
try:
    mock.connect('localhost', 5432)
except ConnectionError as e:
    print('side_effect exception:', e)

# side_effect — different values on successive calls
mock.fetch_page.side_effect = [
    {'page': 1, 'data': [1, 2, 3], 'has_next': True},
    {'page': 2, 'data': [4, 5, 6], 'has_next': True},
    {'page': 3, 'data': [7, 8, 9], 'has_next': False},
]

print('\nPagination simulation:')
all_data = []
page = 1
while True:
    response = mock.fetch_page(page)
    all_data.extend(response['data'])
    if not response['has_next']:
        break
    page += 1
print('All data:', all_data)

# side_effect — function
def validate_and_double(value):
    if value < 0:
        raise ValueError(f'Negative value: {value}')
    return value * 2

mock.process.side_effect = validate_and_double
print('\nside_effect function:')
print('process(5):', mock.process(5))
try:
    mock.process(-1)
except ValueError as e:
    print('process(-1) raised:', e)

# Testing a class that uses external dependencies
class WeatherService:
    def __init__(self, api_client):
        self.api = api_client
    
    def get_temperature(self, city):
        response = self.api.get(f'/weather/{city}')
        if response['status'] != 'ok':
            raise RuntimeError(f'API error: {response["error"]}')
        return response['temperature']

# Test with mock
mock_api = Mock()
mock_api.get.return_value = {'status': 'ok', 'temperature': 22.5}

service = WeatherService(mock_api)
temp = service.get_temperature('London')
print(f'\nTemperature in London: {temp}°C')
mock_api.get.assert_called_once_with('/weather/London')

# Test error case
mock_api.get.return_value = {'status': 'error', 'error': 'City not found'}
try:
    service.get_temperature('Atlantis')
except RuntimeError as e:
    print('Error case:', e)

## 6. TDD Example — Test-Driven Development

In [ ]:
# TDD: Red → Green → Refactor

# STEP 1: Write tests FIRST (they will fail)
def test_roman_numeral():
    """Tests for a Roman numeral converter."""
    assert to_roman(1) == 'I'
    assert to_roman(4) == 'IV'
    assert to_roman(9) == 'IX'
    assert to_roman(14) == 'XIV'
    assert to_roman(40) == 'XL'
    assert to_roman(90) == 'XC'
    assert to_roman(400) == 'CD'
    assert to_roman(900) == 'CM'
    assert to_roman(1994) == 'MCMXCIV'
    assert to_roman(2024) == 'MMXXIV'
    return True

# STEP 2: Implement minimal code to pass
def to_roman(num):
    """Convert integer to Roman numeral."""
    values = [
        (1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'),
        (100, 'C'), (90, 'XC'), (50, 'L'), (40, 'XL'),
        (10, 'X'), (9, 'IX'), (5, 'V'), (4, 'IV'), (1, 'I')
    ]
    result = ''
    for value, numeral in values:
        while num >= value:
            result += numeral
            num -= value
    return result

# STEP 3: Run tests
try:
    test_roman_numeral()
    print('All Roman numeral tests PASSED!')
except AssertionError as e:
    print('FAILED:', e)

# Verify some values
test_values = [1, 4, 9, 14, 40, 90, 400, 900, 1994, 2024]
for n in test_values:
    print(f'  {n:4d} = {to_roman(n)}')